# Day 35 Tutorial：读取实际 v0.3 模板并作出 GNN No-Go

> 本 Notebook 只检查仓库中的实际空白模板，不生成结构、不填假数据、不训练模型。

## Goal

核对 v0.3 的真实 sheet、字段、数据行、实测标签和结构来源字段，形成证据化门槛表，并说明为什么当前粘合剂项目不能启动 GNN。

## Setup

模板是字段讨论稿。空单元格不是 0，列名存在不等于已有数据。Notebook 会从当前目录向上寻找仓库根目录，不保存本机绝对路径。

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

WORKBOOK_NAME = "粘合剂重要化学性质_数据格式_v0.3.xlsx"

def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        workbook = (
            candidate
            / "data"
            / "adhesive"
            / "templates"
            / WORKBOOK_NAME
        )
        if workbook.is_file():
            return candidate
    raise FileNotFoundError("没有找到 v0.3 工作簿")

REPO_ROOT = find_repo_root()
WORKBOOK = (
    REPO_ROOT
    / "data"
    / "adhesive"
    / "templates"
    / WORKBOOK_NAME
)

print("workbook located inside repository")
print("pandas:", pd.__version__)

workbook located inside repository
pandas: 2.3.3


## Steps

### 1. 读取实际 sheet 和第 4 行字段名

`header=3` 表示第 4 行是正式字段名；前三行是标题和填写说明。

In [2]:
raw_sheets = pd.read_excel(
    WORKBOOK,
    sheet_name=None,
    header=None,
    engine="openpyxl",
)
sheet_names = list(raw_sheets)
data = pd.read_excel(
    WORKBOOK,
    sheet_name="01_核心化学性质",
    header=3,
    engine="openpyxl",
)

required_columns = [
    "样品编号",
    "粘合剂化学体系",
    "主树脂/基体名称",
    "固化剂/交联剂名称",
    "实测性能名称",
    "实测性能值",
    "实测性能单位",
]
missing_required = [
    column for column in required_columns if column not in data.columns
]

print("sheets:", sheet_names)
print("core table shape:", data.shape)
print("required columns missing:", missing_required)

sheets: ['01_核心化学性质', '02_字段说明', '03_公开依据']
core table shape: (0, 26)
required columns missing: []


### 2. 检查真实行、实测标签和可追溯结构字段

名称列不能替代结构。只有明确的 SMILES、SDF、InChI、结构文件或结构来源字段才进入结构字段候选；即使将来出现候选，也仍需化学组审核内容。

In [3]:
structure_keywords = (
    "SMILES",
    "SDF",
    "InChI",
    "结构文件",
    "结构来源",
)
structure_columns = [
    str(column)
    for column in data.columns
    if any(keyword.lower() in str(column).lower() for keyword in structure_keywords)
]

sample_count = len(data)
label_count = int(data["实测性能值"].notna().sum())

print("real/template rows:", sample_count)
print("non-missing measured labels:", label_count)
print("traceable structure columns:", structure_columns or "not found")
print("No structure is inferred from resin or curing-agent names.")

real/template rows: 0
non-missing measured labels: 0
traceable structure columns: not found
No structure is inferred from resin or curing-agent names.


### 3. 建立门槛表并作出当前决定

空白模板无法证明样本定义、结构审核、多组分表示、权限或真实项目基线已经完成，所以这些门槛保持 `False`。

In [4]:
# 候选字段或非空值的存在，不足以证明内容、来源和实验协议已经一致。
# 这两个门槛只能在领域人员完成审核后，由正式评审记录改为 True。
label_protocol_verified = False
structure_traceability_verified = False

gate_rows = [
    ("真实样本", f"{sample_count} 行", sample_count > 0, "化学组提供获准真实记录"),
    ("一致实测标签", f"{label_count} 个非空标签；一致性未审核", label_protocol_verified, "冻结目标、单位和测试方法"),
    ("可追溯结构", f"候选字段：{structure_columns or '未发现'}；内容/来源未审核", structure_traceability_verified, "提供经审核 SMILES/SDF 与来源"),
    ("样本与图对象", "空白模板不能确认", False, "确认一行代表什么、图代表什么"),
    ("多组分表示", "尚未定义", False, "确认组分、配比、聚合物和工艺进入方式"),
    ("权限与存储", "公开模板不含授权记录", False, "确认训练、组内共享和公开权限"),
    ("真实项目表格基线", "课程材料不等于本人项目实验", False, "先实际运行传统表格基线"),
]
gate_table = pd.DataFrame(
    gate_rows,
    columns=["门槛", "当前证据", "通过", "下一步"],
)

decision = "No-Go：当前模板为空，且没有可追溯结构与项目建模证据"
display(gate_table)
print("current decision:", decision)

,门槛,当前证据,通过,下一步
0,真实样本,0 行,False,化学组提供获准真实记录
1,一致实测标签,0 个非空标签；一致性未审核,False,冻结目标、单位和测试方法
2,可追溯结构,候选字段：未发现；内容/来源未审核,False,提供经审核 SMILES/SDF 与来源
3,样本与图对象,空白模板不能确认,False,确认一行代表什么、图代表什么
4,多组分表示,尚未定义,False,确认组分、配比、聚合物和工艺进入方式
5,权限与存储,公开模板不含授权记录,False,确认训练、组内共享和公开权限
6,真实项目表格基线,课程材料不等于本人项目实验,False,先实际运行传统表格基线


current decision: No-Go：当前模板为空，且没有可追溯结构与项目建模证据


## Checks

断言锁定当前公开仓库事实。未来获准真实数据到达后，应建立新版本并重新评审，而不是删除这些安全检查来强行得到 Go。

In [5]:
assert sheet_names == [
    "01_核心化学性质",
    "02_字段说明",
    "03_公开依据",
]
assert not missing_required
assert data.shape == (0, 26)
assert sample_count == 0
assert label_count == 0
assert structure_columns == []
assert not gate_table["通过"].any()
assert decision.startswith("No-Go")

print("Checks passed: actual v0.3 is an empty 26-column discussion template.")
print("No model was created or trained.")

Checks passed: actual v0.3 is an empty 26-column discussion template.
No model was created or trained.


## Next Steps

1. 向导师确认首个体系、主要目标和一行样本定义；
2. 请化学组定义结构格式、来源、审核责任和多组分关系；
3. 用 3–5 行获准真实样例验接口，不训练；
4. 首批数据审计后先运行传统表格基线；
5. 再重新做 GNN Go/No-Go。当前正确动作是补充证据，不是跳过门槛。